<a href="https://colab.research.google.com/github/NZW9527/Interview-Notebook/blob/master/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.3: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/CLEAN_ColabFold_AF2/input_fasta/NEW-test-2_individual' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/CLEAN_ColabFold_AF2/results/NEW-test-2' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 1 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = True #@param {type:"boolean"}
use_fast_kernels = False #@param {type:"boolean"}
#@markdown - use fused kernels for faster prediction (experimental)


In [3]:
input_dir='/content/drive/MyDrive/CLEAN_ColabFold_AF2/input_fasta/NEW-test-2_individual'
result_dir='/content/drive/MyDrive/CLEAN_ColabFold_AF2/results/NEW-test-2'
print('ACTIVE_PATHS', input_dir, result_dir)exec("from pathlib import Path\nCODE=Path('/content/drive/MyDrive/CLEAN_ColabFold_AF2/code')\nCODE.mkdir(parents=True,exist_ok=True)\nhelper=\"from pathlib import Path\\nimport csv, json, math, re, zipfile\\nROOT = Path(\\\"/content/drive/MyDrive/CLEAN_ColabFold_AF2\\\")\\nINPUT = ROOT / \\\"input_fasta\\\"\\nCODE = ROOT / \\\"code\\\"\\nJOBS = {\\n    \\\"NEW-test-2\\\": (INPUT / \\\"NEW-test-2.fasta\\\", INPUT / \\\"NEW-test-2_individual\\\", ROOT / \\\"results\\\" / \\\"NEW-test-2\\\", CODE / \\\"NEW-test-2_scores.tsv\\\"),\\n    \\\"split100reduce-62-31\\\": (INPUT / \\\"split100reduce-62-31.fasta\\\", INPUT / \\\"split100reduce-62-31_individual\\\", ROOT / \\\"results\\\" / \\\"split100reduce-62-31\\\", CODE / \\\"split100reduce-62-31_scores.tsv\\\"),\\n}\\ndef read_fasta(path):\\n    records, header, seq = [], None, []\\n    for line in path.read_text(encoding=\\\"utf-8-sig\\\").splitlines():\\n        line = line.strip()\\n        if not line:\\n            continue\\n        if line.startswith(\\\">\\\"):\\n            if header is not None:\\n                records.append((header, \\\"\\\".join(seq)))\\n            header, seq = line[1:].strip(), []\\n        elif header is not None:\\n            seq.append(line)\\n    if header is not None:\\n        records.append((header, \\\"\\\".join(seq)))\\n    return records\\ndef safe_stem(header, index):\\n    raw = header.split()[0] if header else (\\\"record_%04d\\\" % index)\\n    return re.sub(r\\\"[^A-Za-z0-9_.-]+\\\", \\\"_\\\", raw) or (\\\"record_%04d\\\" % index)\\nCODE.mkdir(parents=True, exist_ok=True)\\nfor label, (source, staging, result_dir, score_log) in JOBS.items():\\n    if not source.is_file():\\n        raise FileNotFoundError(\\\"FASTA not found: \\\" + str(source))\\n    records = read_fasta(source)\\n    if not records:\\n        raise ValueError(\\\"No FASTA records in: \\\" + str(source))\\n    staging.mkdir(parents=True, exist_ok=True)\\n    result_dir.mkdir(parents=True, exist_ok=True)\\n    seen = set()\\n    manifest_rows = []\\n    print(\\\"\\\\n\\\" + label + \\\": \\\" + str(len(records)) + \\\" FASTA records\\\")\\n    for i, (header, sequence) in enumerate(records, 1):\\n        if not sequence:\\n            raise ValueError(\\\"Empty sequence: \\\" + header)\\n        base = safe_stem(header, i)\\n        stem, dup = base, 2\\n        while stem in seen:\\n            stem = base + \\\"_dup\\\" + str(dup)\\n            dup += 1\\n        seen.add(stem)\\n        content = \\\">\\\" + header + \\\"\\\\n\\\" + sequence + \\\"\\\\n\\\"\\n        dest = staging / (stem + \\\".fasta\\\")\\n        if dest.exists() and dest.read_text(encoding=\\\"utf-8\\\") != content:\\n            suffix = 2\\n            while (staging / (stem + \\\"_\\\" + str(suffix) + \\\".fasta\\\")).exists():\\n                suffix += 1\\n            dest = staging / (stem + \\\"_\\\" + str(suffix) + \\\".fasta\\\")\\n        if not dest.exists():\\n            dest.write_text(content, encoding=\\\"utf-8\\\")\\n            state = \\\"CREATED\\\"\\n        else:\\n            state = \\\"EXISTS_MATCH\\\"\\n        seq_id = header.split()[0] if header else stem\\n        print(\\\"  ID=\\\" + seq_id + \\\" length=\\\" + str(len(sequence)) + \\\" \\\" + state + \\\" \\\" + dest.name)\\n        manifest_rows.append([seq_id, header, len(sequence), dest.name])\\n    manifest = CODE / (label + \\\"_manifest_20260930.tsv\\\")\\n    if not manifest.exists():\\n        with manifest.open(\\\"x\\\", encoding=\\\"utf-8\\\", newline=\\\"\\\") as f:\\n            w = csv.writer(f, delimiter=\\\"\\\\t\\\")\\n            w.writerow([\\\"sequence_id\\\", \\\"fasta_header\\\", \\\"length\\\", \\\"staged_fasta\\\"])\\n            w.writerows(manifest_rows)\\n    if not score_log.exists():\\n        with score_log.open(\\\"x\\\", encoding=\\\"utf-8\\\", newline=\\\"\\\") as f:\\n            csv.writer(f, delimiter=\\\"\\\\t\\\").writerow([\\\"sequence\\\", \\\"result_zip\\\", \\\"pdb_member\\\", \\\"score_json_member\\\", \\\"mean_plddt\\\", \\\"ptm\\\"])\\n    print(\\\"INPUT_DIR\\\", staging, \\\"RESULT_DIR\\\", result_dir, \\\"SCORE_LOG\\\", score_log)\\ndef log_results(label):\\n    source, staging, result_dir, score_log = JOBS[label]\\n    existing = set()\\n    if score_log.exists():\\n        with score_log.open(encoding=\\\"utf-8\\\", newline=\\\"\\\") as f:\\n            existing = {(r[\\\"result_zip\\\"], r[\\\"pdb_member\\\"]) for r in csv.DictReader(f, delimiter=\\\"\\\\t\\\")}\\n    added = 0\\n    with score_log.open(\\\"a\\\", encoding=\\\"utf-8\\\", newline=\\\"\\\") as out:\\n        writer = csv.writer(out, delimiter=\\\"\\\\t\\\")\\n        for archive in sorted(result_dir.rglob(\\\"*.result.zip\\\")):\\n            seq_id = archive.name[:-len(\\\".result.zip\\\")]\\n            with zipfile.ZipFile(archive) as zf:\\n                names = zf.namelist()\\n                pdbs = sorted(n for n in names if re.search(r\\\"_unrelaxed_rank_001_.*\\\\.pdb$\\\", n, re.I))\\n                scores = sorted(n for n in names if re.search(r\\\"_scores_rank_001_.*\\\\.json$\\\", n, re.I))\\n                print(\\\"SEQ\\\", seq_id, \\\"DIR_EXISTS\\\", archive.parent.exists(), \\\"ZIPS\\\", [(archive.name, archive.stat().st_size)], \\\"ARCHIVE\\\", archive.name, \\\"PDBS\\\", pdbs, \\\"SCORES\\\", scores)\\n                for pdb_member in pdbs:\\n                    if (archive.name, pdb_member) in existing:\\n                        continue\\n                    expected = pdb_member.replace(\\\"_unrelaxed_rank_\\\", \\\"_scores_rank_\\\").rsplit(\\\".\\\", 1)[0] + \\\".json\\\"\\n                    json_member = expected if expected in names else (scores[0] if len(scores) == 1 else None)\\n                    if json_member is None:\\n                        print(\\\" SCORE_JSON_NOT_MATCHED\\\", pdb_member)\\n                        continue\\n                    data = json.loads(zf.read(json_member).decode(\\\"utf-8\\\"))\\n                    mean = data.get(\\\"mean_plddt\\\")\\n                    if not isinstance(mean, (int, float)):\\n                        vals = data.get(\\\"plddt\\\")\\n                        mean = math.fsum(v for v in vals if isinstance(v, (int, float))) / len(vals) if isinstance(vals, list) and vals else None\\n                    ptm = data.get(\\\"ptm\\\")\\n                    if isinstance(mean, (int, float)) and isinstance(ptm, (int, float)):\\n                        print(\\\" SCORE %.3f %.2f\\\" % (mean, ptm))\\n                    else:\\n                        print(\\\" SCORE unavailable mean_plddt=\\\" + str(mean) + \\\" ptm=\\\" + str(ptm))\\n                    writer.writerow([seq_id, archive.name, pdb_member, json_member, \\\"%.6f\\\" % mean if isinstance(mean, (int, float)) else \\\"\\\", \\\"%.6f\\\" % ptm if isinstance(ptm, (int, float)) else \\\"\\\"])\\n                    existing.add((archive.name, pdb_member))\\n                    added += 1\\n    print(label + \\\": appended \\\" + str(added) + \\\" score row(s) to \\\" + str(score_log))\\nprint(\\\"STAGING_READY\\\")\"\np=CODE/'prepare_fasta_scores_20260930.py'\ni=2\nwhile p.exists():\n p=CODE/('prepare_fasta_scores_20260930_v'+str(i)+'.py')\n i+=1\np.write_text(helper,encoding='utf-8')\nprint('HELPER_SAVED',p)\nexec(compile(helper,str(p),'exec'))")

HELPER_SAVED /content/drive/MyDrive/CLEAN_ColabFold_AF2/code/prepare_fasta_scores_20260930.py

NEW-test-2: 2 FASTA records
  ID=O24527 length=836 CREATED O24527.fasta
  ID=Q0E2F9 length=975 CREATED Q0E2F9.fasta
INPUT_DIR /content/drive/MyDrive/CLEAN_ColabFold_AF2/input_fasta/NEW-test-2_individual RESULT_DIR /content/drive/MyDrive/CLEAN_ColabFold_AF2/results/NEW-test-2 SCORE_LOG /content/drive/MyDrive/CLEAN_ColabFold_AF2/code/NEW-test-2_scores.tsv

split100reduce-62-31: 31 FASTA records
  ID=A0A0B4J2F2 length=783 CREATED A0A0B4J2F2.fasta
  ID=P22281 length=533 CREATED P22281.fasta
  ID=Q5XG87 length=772 CREATED Q5XG87.fasta
  ID=Q6PB75 length=542 CREATED Q6PB75.fasta
  ID=A4VCL2 length=980 CREATED A4VCL2.fasta
  ID=Q3UVC0 length=959 CREATED Q3UVC0.fasta
  ID=Q52L14 length=618 CREATED Q52L14.fasta
  ID=P0CR12 length=551 CREATED P0CR12.fasta
  ID=O88623 length=613 CREATED O88623.fasta
  ID=Q8HXX9 length=210 CREATED Q8HXX9.fasta
  ID=Q4I830 length=796 CREATED Q4I830.fasta
  ID=P49842 lengt

In [2]:
#@title Install dependencies
%%bash -s $use_amber

set -e

USE_AMBER=$1

EXTRAS="alphafold-minus-jax"
OPENMM=""
READY=COLABFOLD_READY
if [ ${USE_AMBER} == "True" ]; then
  EXTRAS="alphafold-minus-jax,openmm"
  OPENMM=$(python -c "from importlib.metadata import distributions; i={d.metadata['Name'] for d in distributions()}; print(next((f'openmm[{c}]' for c in ('cuda13','cuda12') if f'jax-{c}-plugin' in i), ''))")
  READY=COLABFOLD_AMBER_READY
fi

if [ ! -f ${READY} ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[${EXTRAS}] @ git+https://github.com/sokrypton/ColabFold" ${OPENMM}
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  touch ${READY}
fi

# Download params (~1min)
python -m colabfold.download


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.0/270.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.9 MB/s eta 0:00:00


In [ ]:
#@title Run Prediction

import sys

from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path

setup_logging(Path(result_dir).joinpath("log.txt"))

queries, is_complex = get_queries(input_dir)
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    num_relax=num_relax,
    relax_max_iterations=relax_max_iterations,
    msa_mode=msa_mode,
    model_type="auto",
    num_models=num_models,
    num_recycles=num_recycles,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=default_data_dir,
    keep_existing_results=do_not_overwrite_results,
    rank_by="auto",
    pair_mode="unpaired+paired",
    stop_at_score=stop_at_score,
    zip_results=zip_results,
    use_fast_kernels=use_fast_kernels,
    user_agent="colabfold/google-colab-batch",
)

2026-09-29 18:05:28,393 Running on GPU
2026-09-29 18:05:29,350 Found 5 citations for tools or databases
2026-09-29 18:05:29,354 Query 1/2: O24527 (length 836)


PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

2026-09-29 18:05:29,978 Sleeping for 10s. Reason: PENDING


RUNNING:   7%|▋         | 10/150 [elapsed: 00:11 remaining: 02:36]

2026-09-29 18:05:40,579 Sleeping for 10s. Reason: RUNNING


...

2026-09-29 18:08:53,970 Sleeping for 6s. Reason: RUNNING


RUNNING: |          | 195/? [elapsed: 03:31 remaining: 00:00]

2026-09-29 18:09:00,570 Sleeping for 5s. Reason: RUNNING


RUNNING: |          | 200/? [elapsed: 03:36 remaining: 00:00]

2026-09-29 18:09:06,167 Sleeping for 10s. Reason: RUNNING


# Instructions <a name="Instructions"></a>
**Quick start**
1. Upload your single fasta files to a folder in your Google Drive
2. Define path to the fold containing the fasta files (`input_dir`) define an outdir (`output_dir`)
3. Press "Runtime" -> "Run all".

**Result zip file contents**

At the end of the job a all results `jobname.result.zip` will be uploaded to your (`output_dir`) Google Drive. Each zip contains one protein.

1. PDB formatted structures sorted by avg. pIDDT. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.


**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and  [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
